In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    root_mean_squared_error, 
    mean_absolute_error,
    mean_absolute_percentage_error
)
import lightgbm as lgb

df = pd.read_csv('data/AmesHousing.csv')

df = df.drop(columns = ['PID', 'Order'])

TARGET = 'SalePrice'

In [20]:
y = np.log1p(df[TARGET])
X = df.drop(columns = [TARGET])

In [21]:
# Convert every non-numeric column to pandas 'category' dtype.
# LightGBM will handle these natively (no one-hot encoding needed) and
# also handles NaNs in both numeric and categorical columns on its own.
cat_cols = X.select_dtypes(include=["object", "str"]).columns.tolist()
 
# A few columns are stored as integers but are really category codes, not
# quantities. Treating them as numeric implies an ordering/magnitude that
# isn't meaningful:
#   - MS SubClass: dwelling-type code (20, 60, 120, ...) -- pure ID, no order
#   - Yr Sold / Mo Sold: low-cardinality (5 and 12 levels) and not
#     necessarily monotonic in price (e.g. a crash year), so letting
#     LightGBM's categorical-split algorithm group arbitrary levels together
#     is preferable to a numeric threshold split
numeric_but_categorical = [c for c in ["MS SubClass", "Yr Sold", "Mo Sold"] if c in X.columns]
cat_cols += numeric_but_categorical

for c in cat_cols:
    X[c] = X[c].astype("category")

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
 
# ---- 4. Train ----
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
)
 
model.fit(
    X_train,
    y_train,
    eval_X = X_test,
    eval_y = y_test,
    eval_metric="rmse",
    categorical_feature=cat_cols,
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)],
)
 
# ---- 5. Evaluate (back on the original $ scale) ----
pred_log = model.predict(X_test, num_iteration=model.best_iteration_)
pred = np.expm1(pred_log)
y_test_dollars = np.expm1(y_test)
 
rmse = root_mean_squared_error(y_test_dollars, pred)
mae = mean_absolute_error(y_test_dollars, pred)
mape = mean_absolute_percentage_error(y_test_dollars, pred)  # fraction, e.g. 0.08 = 8%
ape = np.abs((y_test_dollars - pred) / y_test_dollars)
median_ape = np.median(ape)
 
print(f"\nTest RMSE:        ${rmse:,.0f}")
print(f"Test MAE:         ${mae:,.0f}")
print(f"Test MAPE:        {mape:.2%}")
print(f"Test Median APE:  {median_ape:.2%}")
 
# ---- 6. Feature importance (quick sanity check) ----
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 15 features by importance:")
print(importances.head(15))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000756 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3697
[LightGBM] [Info] Number of data points in the train set: 2344, number of used features: 77
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 12.011814
Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 0.116013	valid_0's l2: 0.0134589
[200]	valid_0's rmse: 0.108845	valid_0's l2: 0.0118472
[300]	valid_0's rmse: 0.10685	valid_0's l2: 0.0114169
[400]	valid_0's rmse: 0.106877	valid_0's l2: 0.0114227
Early stopping, best iteration is:
[343]	valid_0's rmse: 0.106675	valid_0's l2: 0.0113795

Test RMSE:        $22,752
Test MAE:         $14,087
Test MAPE:        7.46%
Test Median APE:  5.24%

Top 15 features by import